<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Ejercicios/Soluciones/04_Taller_HMM_MEMM_Solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Taller práctico: HMM, Viterbi y MEMM (solución)</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Instrucciones</h2>

Solución de referencia del taller [`04_Taller_HMM_MEMM.ipynb`](../04_Taller_HMM_MEMM.ipynb).

Hay muchas soluciones válidas: esta es una de ellas. Compara tus resultados y, sobre todo, tus conclusiones.

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Ejercicio 1. Entidades a partir de etiquetas BIO y evaluación</span>](#ejercicio-1)
- [<span style="color: #005C99;">Ejercicio 2. El algoritmo forward en espacio logarítmico</span>](#ejercicio-2)
- [<span style="color: #005C99;">Ejercicio 3. Un HMM para NER con formas de palabra</span>](#ejercicio-3)
- [<span style="color: #005C99;">Ejercicio 4. Viterbi para NER</span>](#ejercicio-4)
- [<span style="color: #005C99;">Ejercicio 5. Un MEMM para NER</span>](#ejercicio-5)
- [<span style="color: #005C99;">Ejercicio 6. Análisis de errores</span>](#ejercicio-6)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [1]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib scipy scikit-learn nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import itertools
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import logsumexp

import nltk
from nltk.corpus import conll2002

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

SEED = 42
np.random.seed(SEED)

_ = nltk.download("conll2002", quiet=True)


def cargar(particion):
    """Lista de oraciones; cada oración es una lista de (palabra, etiqueta BIO)."""
    return [[(w, ner) for w, _, ner in s] for s in conll2002.iob_sents(particion)]


train, dev, test = cargar("esp.train"), cargar("esp.testa"), cargar("esp.testb")
ETIQUETAS = sorted({t for s in train for _, t in s})
t2i = {t: i for i, t in enumerate(ETIQUETAS)}
print(f"Oraciones: train={len(train)}, dev={len(dev)}, test={len(test)}")
print("Etiquetas:", ETIQUETAS)
print(train[2][:15])

Oraciones: train=8323, dev=1915, test=1517
Etiquetas: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']
[('El', 'O'), ('Abogado', 'B-PER'), ('General', 'I-PER'), ('del', 'I-PER'), ('Estado', 'I-PER'), (',', 'O'), ('Daryl', 'B-PER'), ('Williams', 'I-PER'), (',', 'O'), ('subrayó', 'O'), ('hoy', 'O'), ('la', 'O'), ('necesidad', 'O'), ('de', 'O'), ('tomar', 'O')]


---

<a id="ejercicio-1"></a>
<h2 style="color: #007ACC;">Ejercicio 1. Entidades a partir de etiquetas BIO y evaluación</h2>

**Objetivo:** convertir secuencias BIO en entidades y evaluar con precisión, *recall* y F1 por entidad, como en la tarea compartida CoNLL.

1. Calcula la distribución de etiquetas en entrenamiento. ¿Qué exactitud por token obtendría un etiquetador que siempre predice `O`?
2. Implementa `entidades(tags)` que devuelva el **conjunto** de entidades de una oración como tuplas `(inicio, fin, tipo)` (con `fin` exclusivo). Sigue la convención del script oficial `conlleval`: una entidad empieza con `B-X`, o con `I-X` si la etiqueta anterior no era del mismo tipo `X`.
3. Implementa `evaluar_entidades(oraciones_reales, oraciones_predichas)` que devuelva precisión, *recall* y F1 **micro** (sumando sobre todo el corpus). Una entidad predicha cuenta como correcta solo si coinciden el inicio, el fin **y** el tipo. Devuelve también el F1 por tipo de entidad.
4. Verifica con este ejemplo: reales `B-PER I-PER O B-LOC`, predichas `B-PER O O B-ORG` (precisión 0, recall 0: los límites de la persona y el tipo del lugar están mal).
5. ¿Por qué la exactitud por token es una mala métrica para NER?

In [3]:
conteo = Counter(t for s in train for _, t in s)
print(pd.Series(conteo).sort_values(ascending=False).to_string())
print(f"\nExactitud por token de 'siempre O': {conteo['O'] / sum(conteo.values()):.3f}")

O         231920
B-ORG       7390
I-ORG       4992
B-LOC       4913
B-PER       4321
I-PER       3903
I-MISC      3212
B-MISC      2173
I-LOC       1891

Exactitud por token de 'siempre O': 0.876


In [4]:
def entidades(tags):
    ents, tipo, inicio = set(), None, None
    for i, t in enumerate(list(tags) + ["O"]):
        continua = t.startswith("I-") and t[2:] == tipo
        if not continua:
            if tipo is not None:
                ents.add((inicio, i, tipo))
                tipo = None
            if t != "O":
                tipo, inicio = t[2:], i
    return ents


def evaluar_entidades(reales, predichas):
    tp, n_pred, n_real = Counter(), Counter(), Counter()
    for r, p in zip(reales, predichas):
        er, ep = entidades(r), entidades(p)
        for _, _, tipo in ep:
            n_pred[tipo] += 1
        for _, _, tipo in er:
            n_real[tipo] += 1
        for _, _, tipo in er & ep:
            tp[tipo] += 1

    def prf(tp_, np_, nr_):
        prec = tp_ / np_ if np_ else 0.0
        rec = tp_ / nr_ if nr_ else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        return prec, rec, f1

    total = prf(sum(tp.values()), sum(n_pred.values()), sum(n_real.values()))
    por_tipo = {t: prf(tp[t], n_pred[t], n_real[t])[2] for t in sorted(n_real)}
    return {"precisión": total[0], "recall": total[1], "F1": total[2], **{f"F1 {t}": f for t, f in por_tipo.items()}}


print(entidades(["B-PER", "I-PER", "O", "B-LOC", "I-LOC", "O", "I-ORG"]))
print(evaluar_entidades([["B-PER", "I-PER", "O", "B-LOC"]], [["B-PER", "O", "O", "B-ORG"]]))

{(6, 7, 'ORG'), (0, 2, 'PER'), (3, 5, 'LOC')}
{'precisión': 0.0, 'recall': 0.0, 'F1': 0.0, 'F1 LOC': 0.0, 'F1 PER': 0.0}


**Interpretación:** casi el 90 % de los tokens son `O`, así que un etiquetador que nunca encuentra ninguna entidad ya tiene ≈88 % de exactitud por token, y su F1 por entidades es 0. Además, la exactitud por token da crédito parcial a una entidad con los límites equivocados (*"Banco de España"* detectado solo como *"Banco"*), cuando para cualquier aplicación esa entidad está mal. Por eso NER se evalúa **por entidades completas**.

---

<a id="ejercicio-2"></a>
<h2 style="color: #007ACC;">Ejercicio 2. El algoritmo forward en espacio logarítmico</h2>

**Objetivo:** implementar forward con log-probabilidades y comprobar por qué es necesario.

Usa este HMM de juguete con 3 estados y 4 símbolos:

```python
pi_j = np.array([0.6, 0.3, 0.1])
A_j = np.array([[0.7, 0.2, 0.1], [0.3, 0.5, 0.2], [0.2, 0.3, 0.5]])
B_j = np.array([[0.5, 0.3, 0.1, 0.1], [0.1, 0.4, 0.4, 0.1], [0.1, 0.1, 0.2, 0.6]])
```

1. Implementa `forward(obs, pi, A, B)` en probabilidades (como en el cuaderno 08) y `forward_log(obs, log_pi, log_A, log_B)` usando `scipy.special.logsumexp`. Pista: la recursión es $\log \alpha_t(j) = \text{logsumexp}_i\left(\log \alpha_{t-1}(i) + \log a_{ij}\right) + \log b_j(o_t)$.
2. Verifica ambas contra la fuerza bruta para una secuencia de longitud 6.
3. Genera una secuencia aleatoria de longitud 1 000 y calcula $P(O)$ con ambas versiones. ¿Qué ocurre y por qué?

In [5]:
pi_j = np.array([0.6, 0.3, 0.1])
A_j = np.array([[0.7, 0.2, 0.1], [0.3, 0.5, 0.2], [0.2, 0.3, 0.5]])
B_j = np.array([[0.5, 0.3, 0.1, 0.1], [0.1, 0.4, 0.4, 0.1], [0.1, 0.1, 0.2, 0.6]])


def forward(obs, pi, A, B):
    alpha = pi * B[:, obs[0]]
    for o in obs[1:]:
        alpha = (alpha @ A) * B[:, o]
    return alpha.sum()


def forward_log(obs, log_pi, log_A, log_B):
    log_alpha = log_pi + log_B[:, obs[0]]
    for o in obs[1:]:
        log_alpha = logsumexp(log_alpha[:, None] + log_A, axis=0) + log_B[:, o]
    return logsumexp(log_alpha)


def fuerza_bruta(obs, pi, A, B):
    total = 0.0
    for q in itertools.product(range(len(pi)), repeat=len(obs)):
        p = pi[q[0]] * B[q[0], obs[0]]
        for t in range(1, len(obs)):
            p *= A[q[t - 1], q[t]] * B[q[t], obs[t]]
        total += p
    return total


rng = np.random.default_rng(SEED)
obs = list(rng.integers(0, 4, size=6))
logs = (np.log(pi_j), np.log(A_j), np.log(B_j))
print(f"fuerza bruta: {fuerza_bruta(obs, pi_j, A_j, B_j):.6e}")
print(f"forward:      {forward(obs, pi_j, A_j, B_j):.6e}")
print(f"forward_log:  {np.exp(forward_log(obs, *logs)):.6e}")

fuerza bruta: 1.889328e-04
forward:      1.889328e-04
forward_log:  1.889328e-04


In [6]:
obs_largo = list(rng.integers(0, 4, size=1000))
print(f"forward (probabilidades): P(O) = {forward(obs_largo, pi_j, A_j, B_j)}")
print(f"forward_log:              log P(O) = {forward_log(obs_largo, *logs):.2f}")

forward (probabilidades): P(O) = 0.0
forward_log:              log P(O) = -1407.72


**Interpretación:** con 1 000 observaciones, $P(O)$ es del orden de $10^{-600}$, muy por debajo del menor número positivo representable en punto flotante de doble precisión (≈$10^{-308}$). La versión en probabilidades da exactamente **0** (*underflow*) y toda la información se pierde. La versión logarítmica trabaja con números del orden de $-1\,400$, perfectamente representables. `logsumexp` calcula $\log \sum_i e^{x_i}$ de forma estable restando el máximo antes de exponenciar.

---

<a id="ejercicio-3"></a>
<h2 style="color: #007ACC;">Ejercicio 3. Un HMM para NER con formas de palabra</h2>

**Objetivo:** estimar un HMM supervisado para NER en el que las palabras raras y desconocidas se representan por su forma.

En NER, la **forma** de una palabra es una pista muy fuerte: *Madrid* (`Xx`), *ONU* (`X`), *1998* (`d`), *Juan-Carlos* (`Xx-Xx`)...

1. Implementa `forma(palabra)` que reemplace las mayúsculas por `X`, las minúsculas por `x` y los dígitos por `d`, colapsando las repeticiones consecutivas: `"Madrid" → "Xx"`, `"ONU" → "X"`, `"1998" → "d"`, `"EE.UU." → "X.X."`, `"Juan-Carlos" → "Xx-Xx"`. Añade el prefijo `"<FORMA>"` para que no se confunda con una palabra real.
2. Define el vocabulario como las palabras (respetando mayúsculas) que aparecen al menos 2 veces en entrenamiento. Las demás, en entrenamiento y en evaluación, se reemplazan por su forma.
3. Estima $\pi$, $A$ y $B$ por conteo con un suavizado aditivo pequeño (por ejemplo $10^{-3}$) y guarda sus logaritmos.
4. Muestra las 5 transiciones más probables desde `B-PER`, `B-ORG` y `O`. ¿Aprendió el modelo la gramática del formato BIO (por ejemplo, que `I-PER` solo sigue a `B-PER` o `I-PER`)?
5. Muestra las formas más frecuentes que emite cada tipo de entidad.

In [7]:
def forma(palabra):
    s = "".join("X" if c.isupper() else "x" if c.islower() else "d" if c.isdigit() else c for c in palabra)
    colapsada = "".join(c for i, c in enumerate(s) if i == 0 or c != s[i - 1])
    return "<FORMA>" + colapsada


for w in ["Madrid", "ONU", "1998", "EE.UU.", "Juan-Carlos", "casa"]:
    print(f"{w:<12} -> {forma(w)}")

Madrid       -> <FORMA>Xx
ONU          -> <FORMA>X
1998         -> <FORMA>d
EE.UU.       -> <FORMA>X.X.
Juan-Carlos  -> <FORMA>Xx-Xx
casa         -> <FORMA>x


In [8]:
frec = Counter(w for s in train for w, _ in s)
VOCAB = {w for w, c in frec.items() if c >= 2}


def normalizar(palabras):
    return [w if w in VOCAB else forma(w) for w in palabras]


N = len(ETIQUETAS)
c_ini, c_trans = np.zeros(N), np.zeros((N, N))
c_emis = defaultdict(Counter)
for s in train:
    palabras, tags = zip(*s)
    ids = [t2i[t] for t in tags]
    c_ini[ids[0]] += 1
    for i, j in zip(ids, ids[1:]):
        c_trans[i, j] += 1
    for w, j in zip(normalizar(palabras), ids):
        c_emis[j][w] += 1

alpha = 1e-3
log_pi = np.log((c_ini + alpha) / (c_ini + alpha).sum())
log_A = np.log((c_trans + alpha) / (c_trans + alpha).sum(axis=1, keepdims=True))
V = len({w for j in c_emis for w in c_emis[j]})
total_emis = np.array([sum(c_emis[j].values()) for j in range(N)])


def log_B_oracion(palabras):
    return np.array([[np.log((c_emis[j][w] + alpha) / (total_emis[j] + alpha * V)) for j in range(N)]
                     for w in normalizar(palabras)])


A_df = pd.DataFrame(np.exp(log_A), index=ETIQUETAS, columns=ETIQUETAS)
for origen in ["B-PER", "B-ORG", "O"]:
    print(f"{origen:<6} ->", A_df.loc[origen].sort_values(ascending=False).head(5).round(3).to_dict())
print("\nP(I-PER | O) =", round(A_df.loc["O", "I-PER"], 6), "| P(I-PER | B-LOC) =", round(A_df.loc["B-LOC", "I-PER"], 6))

B-PER  -> {'I-PER': 0.666, 'O': 0.333, 'B-LOC': 0.001, 'B-ORG': 0.0, 'B-PER': 0.0}
B-ORG  -> {'O': 0.686, 'I-ORG': 0.303, 'B-PER': 0.006, 'B-LOC': 0.004, 'B-MISC': 0.0}
O      -> {'O': 0.924, 'B-ORG': 0.032, 'B-LOC': 0.019, 'B-PER': 0.016, 'B-MISC': 0.009}

P(I-PER | O) = 0.0 | P(I-PER | B-LOC) = 0.0


In [9]:
pd.DataFrame({
    t: [f"{w} ({c})" for w, c in Counter({w: c for w, c in c_emis[t2i[t]].items() if w.startswith("<FORMA>")}).most_common(4)]
    for t in ["B-PER", "B-LOC", "B-ORG", "B-MISC", "O"]
})

,B-PER,B-LOC,B-ORG,B-MISC,O
0,<FORMA>Xx (438),<FORMA>Xx (317),<FORMA>X (205),<FORMA>Xx (177),<FORMA>x (6800)
1,<FORMA>X (39),<FORMA>X (28),<FORMA>Xx (202),<FORMA>X (90),<FORMA>d.d (474)
2,<FORMA>Xx-Xx (4),<FORMA>Xx-Xx (9),<FORMA>X.X (30),<FORMA>Xd (39),<FORMA>X (469)
3,<FORMA>x (4),<FORMA>x (3),<FORMA>Xx-Xx (9),<FORMA>X-d (15),"<FORMA>d,d (332)"


**Interpretación:**

- El modelo aprendió la gramática BIO solo contando: las transiciones inválidas, como `O → I-PER` o `B-LOC → I-PER`, tienen probabilidad prácticamente nula (solo la del suavizado). Desde `B-PER` lo más probable es `I-PER` (nombre y apellido) o `O`.
- Las formas emitidas por las entidades empiezan casi siempre con mayúscula (`Xx`, `X`), mientras que `O` emite sobre todo formas en minúscula o numéricas. La forma de las palabras raras aporta así la señal que más necesita un etiquetador de entidades.

---

<a id="ejercicio-4"></a>
<h2 style="color: #007ACC;">Ejercicio 4. Viterbi para NER</h2>

**Objetivo:** implementar Viterbi en espacio logarítmico, decodificar con el HMM y compararlo con una línea base.

1. Implementa `viterbi(log_pi, log_A, log_B)` donde `log_B` es la matriz $(T \times N)$ de log-emisiones de la oración. Devuelve la lista de índices de estados.
2. Implementa `etiquetar_hmm(palabras)` y etiqueta el conjunto de **validación**.
3. Implementa una **línea base**: cada palabra recibe la etiqueta con la que más apareció en entrenamiento; las desconocidas reciben `O`.
4. Compara ambos con la exactitud por token y con `evaluar_entidades` (F1 global y por tipo).
5. Cuenta cuántas transiciones BIO inválidas (`I-X` después de algo que no sea `B-X` o `I-X`) produce cada método. ¿Por qué el HMM casi no las produce?

In [10]:
def viterbi(log_pi, log_A, log_B):
    T, N = log_B.shape
    log_v = log_pi + log_B[0]
    bp = np.zeros((T, N), dtype=int)
    for t in range(1, T):
        puntajes = log_v[:, None] + log_A
        bp[t] = puntajes.argmax(axis=0)
        log_v = puntajes.max(axis=0) + log_B[t]
    camino = [int(log_v.argmax())]
    for t in range(T - 1, 0, -1):
        camino.append(int(bp[t, camino[-1]]))
    return camino[::-1]


def etiquetar_hmm(palabras):
    return [ETIQUETAS[i] for i in viterbi(log_pi, log_A, log_B_oracion(palabras))]


por_palabra = defaultdict(Counter)
for s in train:
    for w, t in s:
        por_palabra[w][t] += 1


def etiquetar_base(palabras):
    return [por_palabra[w].most_common(1)[0][0] if w in por_palabra else "O" for w in palabras]


def invalidas(tags):
    return sum(t.startswith("I-") and (i == 0 or tags[i - 1][2:] != t[2:]) for i, t in enumerate(tags))


def comparar(etiquetadores, oraciones):
    reales = [[t for _, t in s] for s in oraciones]
    filas = {}
    for nombre, f in etiquetadores.items():
        inicio = time.perf_counter()
        pred = [f([w for w, _ in s]) for s in oraciones]
        acc = np.mean([p == r for ps, rs in zip(pred, reales) for p, r in zip(ps, rs)])
        filas[nombre] = {"exactitud token": acc, **evaluar_entidades(reales, pred),
                         "I- inválidas": sum(invalidas(p) for p in pred), "tiempo (s)": time.perf_counter() - inicio}
    return pd.DataFrame(filas).T


resultados_dev = comparar({"línea base": etiquetar_base, "HMM + Viterbi": etiquetar_hmm}, dev)
resultados_dev.round(3)

,exactitud token,precisión,recall,F1,F1 LOC,F1 MISC,F1 ORG,F1 PER,I- inválidas,tiempo (s)
línea base,0.919,0.449,0.437,0.443,0.609,0.252,0.469,0.307,748.0,0.035
HMM + Viterbi,0.937,0.612,0.609,0.610,0.583,0.393,0.650,0.683,1.0,0.556


**Interpretación:**

- La línea base tiene una exactitud por token alta (por la abundancia de `O`), pero un F1 por entidades bajo: no reconoce entidades nuevas y fragmenta las de varias palabras.
- El HMM mejora mucho el F1: las formas de palabra le permiten detectar nombres propios que nunca vio, y las transiciones le dan entidades de varias palabras bien delimitadas.
- La línea base produce muchas secuencias inválidas (`I-X` sin `B-X` previo), porque decide cada palabra por separado. El HMM casi no las produce porque esas transiciones tienen probabilidad prácticamente nula en $A$, y Viterbi evita los caminos que las usan.
- `MISC` es el tipo más difícil: agrupa eventos, obras y nacionalidades con formas muy variadas. Llama la atención que en `LOC` el HMM queda ligeramente **por debajo** de la línea base. Una explicación probable: muchos lugares (*Madrid*, *España*) son palabras frecuentes que la línea base memoriza bien, mientras que el HMM puede arrastrarlas a otro tipo por las transiciones (por ejemplo, después de una preposición en un contexto típico de organizaciones).

---

<a id="ejercicio-5"></a>
<h2 style="color: #007ACC;">Ejercicio 5. Un MEMM para NER</h2>

**Objetivo:** entrenar un MEMM con características ricas, decodificarlo con Viterbi y compararlo con el HMM.

1. Diseña `caracteristicas(palabras, i, etiqueta_previa)` para NER. Incluye al menos: la palabra en minúscula, su forma, prefijos y sufijos, si es la primera palabra de la oración, las palabras vecinas (y si empiezan con mayúscula) y la etiqueta anterior.
2. Entrena un `LogisticRegression` (con *teacher forcing*: la etiqueta anterior verdadera) sobre el conjunto de entrenamiento.
3. Implementa la decodificación con Viterbi del MEMM (como en el cuaderno 10) y etiqueta el conjunto de **validación**.
4. Compara con la línea base y el HMM usando la función `comparar` del ejercicio 4.
5. Elige el mejor modelo según **validación** y evalúalo **una sola vez** en el conjunto de **prueba** (`esp.testb`).

In [11]:
def caracteristicas(palabras, i, etiqueta_previa):
    w = palabras[i]
    p = w.lower()
    f = {
        "palabra=" + p: 1,
        "forma=" + forma(w): 1,
        "suf3=" + p[-3:]: 1,
        "pre3=" + p[:3]: 1,
        "etiqueta_previa=" + etiqueta_previa: 1,
        "previa+forma=" + etiqueta_previa + "|" + forma(w): 1,
    }
    if i == 0:
        f["inicio_oracion"] = 1
    for d, nombre in [(-1, "anterior"), (1, "siguiente")]:
        j = i + d
        if 0 <= j < len(palabras):
            f[f"palabra_{nombre}=" + palabras[j].lower()] = 1
            if palabras[j][:1].isupper():
                f[f"mayúscula_{nombre}"] = 1
        else:
            f[f"palabra_{nombre}=<borde>"] = 1
    return f


X_dict, y = [], []
for s in train:
    palabras, tags = zip(*s)
    for i in range(len(palabras)):
        X_dict.append(caracteristicas(palabras, i, tags[i - 1] if i > 0 else "<S>"))
        y.append(tags[i])

vec = DictVectorizer()
X = vec.fit_transform(X_dict)
inicio = time.perf_counter()
memm = LogisticRegression(C=1.0, max_iter=300).fit(X, y)
CLASES = list(memm.classes_)
print(f"{X.shape[0]:,} ejemplos, {X.shape[1]:,} características, entrenado en {time.perf_counter() - inicio:.1f} s")

264,715 ejemplos, 78,905 características, entrenado en 11.2 s


In [12]:
def etiquetar_memm(palabras):
    T, N = len(palabras), len(CLASES)
    filas = [caracteristicas(palabras, 0, "<S>")]
    filas += [caracteristicas(palabras, t, previa) for t in range(1, T) for previa in CLASES]
    log_p = np.log(memm.predict_proba(vec.transform(filas)) + 1e-12)
    log_v = log_p[0]
    bp = np.zeros((T, N), dtype=int)
    for t in range(1, T):
        puntajes = log_v[:, None] + log_p[1 + (t - 1) * N: 1 + t * N]
        bp[t] = puntajes.argmax(axis=0)
        log_v = puntajes.max(axis=0)
    camino = [int(log_v.argmax())]
    for t in range(T - 1, 0, -1):
        camino.append(int(bp[t, camino[-1]]))
    return [CLASES[i] for i in camino[::-1]]


etiquetadores = {"línea base": etiquetar_base, "HMM + Viterbi": etiquetar_hmm, "MEMM + Viterbi": etiquetar_memm}
comparar(etiquetadores, dev).round(3)

,exactitud token,precisión,recall,F1,F1 LOC,F1 MISC,F1 ORG,F1 PER,I- inválidas,tiempo (s)
línea base,0.919,0.449,0.437,0.443,0.609,0.252,0.469,0.307,748.0,0.056
HMM + Viterbi,0.937,0.612,0.609,0.610,0.583,0.393,0.650,0.683,1.0,0.565
MEMM + Viterbi,0.953,0.743,0.709,0.725,0.695,0.550,0.732,0.809,0.0,1.900


In [13]:
print("Evaluación final en el conjunto de PRUEBA (esp.testb) del modelo elegido en validación:")
comparar({"MEMM + Viterbi": etiquetar_memm}, test).round(3)

Evaluación final en el conjunto de PRUEBA (esp.testb) del modelo elegido en validación:


,exactitud token,precisión,recall,F1,F1 LOC,F1 MISC,F1 ORG,F1 PER,I- inválidas,tiempo (s)
MEMM + Viterbi,0.968,0.779,0.761,0.77,0.775,0.521,0.776,0.85,0.0,1.83


**Interpretación:** el MEMM supera al HMM en F1 por entidades en **todos** los tipos, con las mayores ganancias en `MISC` y `PER`. Combina la forma de la palabra con las palabras vecinas (*"el presidente X"*, *"la Universidad de..."*) y con la etiqueta anterior, algo que el HMM no podía hacer.

En prueba el F1 resulta incluso algo **mayor** que en validación. Es un fenómeno conocido de CoNLL-2002 en español: `esp.testb` es algo más fácil que `esp.testa`. Lo importante es que no usamos la prueba para tomar ninguna decisión. Como referencia, el mejor sistema de la competencia CoNLL-2002 en español (Carreras et al., con características y recursos adicionales) obtuvo un F1 de 0.81 en `esp.testb`.

---

<a id="ejercicio-6"></a>
<h2 style="color: #007ACC;">Ejercicio 6. Análisis de errores</h2>

**Objetivo:** clasificar los errores de NER por tipo y entender qué le falta al modelo.

Con las predicciones del MEMM sobre el conjunto de **validación**:

1. Clasifica cada entidad real no encontrada exactamente en una de estas categorías:
   - **tipo equivocado:** hay una entidad predicha con los mismos límites pero otro tipo;
   - **límites equivocados:** hay una entidad predicha que se solapa con la real pero con otros límites;
   - **omitida:** ninguna entidad predicha se solapa con la real.
2. Cuenta también los **falsos positivos** puros: entidades predichas que no se solapan con ninguna real.
3. Para los errores de tipo, construye una tabla real × predicho. ¿Qué confusiones dominan?
4. Muestra 8 ejemplos de errores con su contexto y comenta al menos tres.

In [14]:
def se_solapan(a, b):
    return a[0] < b[1] and b[0] < a[1]


categorias = Counter()
confusion_tipo = Counter()
ejemplos_error = []
for s in dev:
    palabras = [w for w, _ in s]
    reales_s = entidades([t for _, t in s])
    pred_s = entidades(etiquetar_memm(palabras))
    for e in reales_s - pred_s:
        mismos_limites = [p for p in pred_s if p[:2] == e[:2]]
        solapadas = [p for p in pred_s if se_solapan(p, e)]
        if mismos_limites:
            categorias["tipo equivocado"] += 1
            confusion_tipo[(e[2], mismos_limites[0][2])] += 1
            pred = mismos_limites[0]
        elif solapadas:
            categorias["límites equivocados"] += 1
            pred = solapadas[0]
        else:
            categorias["omitida"] += 1
            pred = None
        ejemplos_error.append((" ".join(palabras[max(0, e[0] - 4):e[1] + 4]), " ".join(palabras[e[0]:e[1]]), e[2],
                               None if pred is None else f"{' '.join(palabras[pred[0]:pred[1]])} ({pred[2]})"))
    categorias["falso positivo"] += sum(1 for p in pred_s if not any(se_solapan(p, e) for e in reales_s))

print(pd.Series(categorias).sort_values(ascending=False).to_string())
tabla = pd.Series(confusion_tipo).unstack(fill_value=0)
tabla.index.name, tabla.columns.name = "real", "predicho"
tabla

tipo equivocado        686
límites equivocados    432
omitida                150
falso positivo          57


predicho,LOC,MISC,ORG,PER
real,,,,
LOC,0,6,86,25
MISC,19,0,56,16
ORG,183,42,0,62
PER,151,2,38,0


In [15]:
pd.set_option("display.max_colwidth", 80)
rng = np.random.default_rng(SEED)
muestra = [ejemplos_error[i] for i in rng.choice(len(ejemplos_error), size=8, replace=False)]
pd.DataFrame(muestra, columns=["contexto", "entidad real", "tipo real", "predicción"])

,contexto,entidad real,tipo real,predicción
0,de Minas. Energía de Brasil al señalar que espera,Brasil,LOC,Minas. Energía de Brasil (MISC)
1,"mundial ) en el estadio del Milenio , por 3-0 ,",estadio del Milenio,LOC,Milenio (MISC)
2,"de ambos grupos , Banamex Accival ( BANACCI ) presentó",Banamex Accival,ORG,Banamex Accival (PER)
3,Barcelona 89 - TAU Vitoria,Barcelona,ORG,Barcelona (LOC)
4,"recibieron la recomendación de Defensa Civil de mantener la """,Defensa Civil,ORG,Defensa Civil (MISC)
5,", una en el Recinto Ferial que dé servicio a",Recinto Ferial,LOC,Recinto Ferial (ORG)
6,por el compromiso de Petrobras para instalar una central,Petrobras,ORG,Petrobras (LOC)
7,"El presidente del Tribunal Superior de Justicia de Andalucía , Augusto Ménde...",Tribunal Superior de Justicia de Andalucía,MISC,Tribunal Superior de Justicia de Andalucía (ORG)


**Interpretación (los detalles dependen de la muestra):**

- Los errores de **tipo** son la categoría más frecuente. La confusión dominante es `ORG` predicha como `LOC`: organizaciones que llevan nombre de lugar, como selecciones y clubes (*Brasil*, *TAU Vitoria*), gobiernos o empresas regionales. Distinguirlas requiere conocimiento del mundo o un contexto más amplio que una palabra a cada lado.
- También es muy frecuente `PER` predicha como `LOC`: probablemente porque, ante un nombre propio desconocido de una sola palabra, el modelo tiene pocas pistas para distinguir una persona de un lugar. Vale la pena revisar ejemplos concretos para confirmarlo.
- Los errores de **límites** son típicos de entidades largas con preposiciones y minúsculas internas (*Ministerio de Asuntos Exteriores*, *Comisión Nacional del Mercado de Valores*): el modelo corta la entidad en una preposición o la extiende de más.
- Las **omisiones** suelen ser entidades en minúscula, siglas poco frecuentes o entidades `MISC` (eventos, obras, nacionalidades) cuya forma no es la de un nombre propio.
- Algunos "errores" son discutibles y reflejan inconsistencias de la anotación.

**Mejoras posibles:** listas de nombres conocidos (*gazetteers*), características de una ventana más amplia, un CRF en lugar de un MEMM (cuaderno 10) y, sobre todo, **embeddings** y modelos neuronales que aprenden representaciones de contexto (módulos 3 y 4).

<a id="referencias"></a>
<h2 style="color: #007ACC;">Referencias</h2>

- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador). [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/)
- Tjong Kim Sang, E. F. (2002). Introduction to the CoNLL-2002 Shared Task: Language-Independent Named Entity Recognition. *CoNLL*.
- Carreras, X., Màrquez, L. & Padró, L. (2002). Named Entity Extraction using AdaBoost. *CoNLL*.
- McCallum, A., Freitag, D. & Pereira, F. (2000). Maximum Entropy Markov Models for Information Extraction and Segmentation. *ICML*.